# 8.1 参数高效微调（PEFT）

在端侧设备上对模型进行微调和个性化适配，使模型更好地服务本地用户，同时保护用户数据隐私。参数高效微调（PEFT）仅训练极少量参数，大幅降低端侧训练的显存需求。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### LoRA / QLoRA

冻结预训练权重，仅训练低秩适配器矩阵。QLoRA进一步将基座模型量化为4bit，仅适配器保持高精度。

In [ ]:
class LoRALinear(nn.Module):
    """LoRA线性层"""
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01, requires_grad=False)
        self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank
        self.merged = False

    def merge(self):
        if not self.merged:
            self.weight.data += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True

    def forward(self, x):
        if self.merged:
            return F.linear(x, self.weight, self.bias)
        base = F.linear(x, self.weight, self.bias)
        lora = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base + lora

class LoRAModel(nn.Module):
    """LoRA微调模型"""
    def __init__(self, dim=64, n_classes=10, rank=8, alpha=16):
        super().__init__()
        self.fc1 = LoRALinear(dim, dim*2, rank, alpha)
        self.fc2 = LoRALinear(dim*2, dim, rank, alpha)
        self.fc3 = LoRALinear(dim, n_classes, rank, alpha)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

    def lora_params(self):
        return [p for n, p in self.named_parameters() if 'lora_' in n]

    def freeze_base(self):
        for n, p in self.named_parameters():
            if 'lora_' not in n:
                p.requires_grad = False

dim, n_classes = 64, 10
model = LoRAModel(dim, n_classes, rank=8, alpha=16)
model.freeze_base()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.lora_params())

x_data = torch.randn(256, dim)
y_data = torch.randint(0, n_classes, (256,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=32)

optimizer = torch.optim.Adam(model.lora_params(), lr=1e-3)
losses = []
for epoch in range(20):
    for x, y in loader:
        loss = F.cross_entropy(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

with torch.no_grad():
    acc_before_merge = (model(x_data).argmax(1) == y_data).float().mean()

model.fc1.merge()
model.fc2.merge()
model.fc3.merge()

with torch.no_grad():
    acc_after_merge = (model(x_data).argmax(1) == y_data).float().mean()

print(f"=== LoRA微调 ===")
print(f"总参数: {total_params:,}")
print(f"可训练参数: {trainable_params:,} ({trainable_params/total_params:.2%})")
print(f"冻结参数: {total_params - trainable_params:,}")
print(f"\n合并前准确率: {acc_before_merge:.4f}")
print(f"合并后准确率: {acc_after_merge:.4f}")
print(f"训练损失: {losses[0]:.4f} -> {losses[-1]:.4f}")

print(f"\n--- 不同rank的参数量 ---")
for r in [1, 2, 4, 8, 16, 32]:
    m = LoRAModel(dim, n_classes, rank=r)
    tp = sum(p.numel() for p in m.parameters())
    lp = sum(p.numel() for p in m.lora_params())
    print(f"  rank={r:<3} LoRA参数={lp:<8} 占比={lp/tp:.2%}")

### Adapter & Prefix Tuning

In [ ]:
class AdapterLayer(nn.Module):
    """Adapter层: 插入在Transformer层中的小型适配器"""
    def __init__(self, dim, bottleneck=16):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck)
        self.up = nn.Linear(bottleneck, dim)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return x + self.gate * self.up(F.relu(self.down(x)))

class PrefixTuning(nn.Module):
    """Prefix Tuning: 在输入前添加可训练的虚拟token"""
    def __init__(self, dim, prefix_length=10):
        super().__init__()
        self.prefix = nn.Parameter(torch.randn(prefix_length, dim) * 0.02)

    def forward(self, x):
        prefix = self.prefix.unsqueeze(0).expand(x.shape[0], -1, -1)
        return torch.cat([prefix, x], dim=1)

class AdapterTransformer(nn.Module):
    """带Adapter的Transformer"""
    def __init__(self, dim=64, n_layers=4, n_heads=4, adapter_bottleneck=16):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'attn': nn.MultiheadAttention(dim, n_heads, batch_first=True),
                'ffn': nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)),
                'adapter': AdapterLayer(dim, adapter_bottleneck),
                'ln1': nn.LayerNorm(dim),
                'ln2': nn.LayerNorm(dim),
            }))

    def forward(self, x):
        for layer in self.layers:
            h = layer['ln1'](x)
            h, _ = layer['attn'](h, h, h)
            x = x + h
            x = x + layer['adapter'](layer['ffn'](layer['ln2'](x)))
        return x

dim = 64
adapter_model = AdapterTransformer(dim, n_layers=4, n_heads=4, adapter_bottleneck=16)
prefix_tuner = PrefixTuning(dim, prefix_length=10)

adapter_params = sum(p.numel() for n, p in adapter_model.named_parameters() if 'adapter' in n)
total_adapter_params = sum(p.numel() for p in adapter_model.parameters())
prefix_params = sum(p.numel() for p in prefix_tuner.parameters())

x = torch.randn(2, 16, dim)
x_with_prefix = prefix_tuner(x)
out = adapter_model(x_with_prefix)

print(f"=== Adapter vs Prefix Tuning ===")
print(f"\nAdapter:")
print(f"  Adapter参数: {adapter_params:,} ({adapter_params/total_adapter_params:.2%})")
print(f"  瓶颈维度: 16")
print(f"  结构: 下投影->ReLU->上投影")
print(f"  门控初始化为0，训练初期不影响原始输出")

print(f"\nPrefix Tuning:")
print(f"  Prefix参数: {prefix_params:,}")
print(f"  Prefix长度: 10")
print(f"  输入形状: {x.shape} -> {x_with_prefix.shape}")

print(f"\n=== PEFT方法对比 ===")
methods = [
    ('LoRA', '低秩矩阵', f'rank×(m+n)', '合并后零开销'),
    ('Adapter', '瓶颈MLP', f'2×dim×bn', '额外前向计算'),
    ('Prefix Tuning', '虚拟token', f'len×dim', '增加序列长度'),
    ('QLoRA', '量化+LoRA', f'rank×(m+n)', '合并后零开销'),
]
print(f"\n{'方法':<15} {'原理':<15} {'参数量':<15} {'推理开销':<15}")
print("-" * 60)
for name, principle, params, overhead in methods:
    print(f"{name:<15} {principle:<15} {params:<15} {overhead:<15}")

### QLoRA：量化基座 + LoRA微调

QLoRA将基座模型量化为4bit（NF4格式），仅LoRA适配器保持高精度，使7B模型在24GB显存上可训练。核心创新：NF4量化格式、双重量化（double quantization）和分页优化器。

In [ ]:
class NF4Quantizer:
    """NF4（Normal Float 4-bit）量化器
    NF4假设权重服从正态分布，将量化点均匀分布在正态分布的分位数上，
    使得每个量化区间包含大致相同的概率质量，比均匀量化精度更高。"""
    def __init__(self, n_bins=16):
        self.n_bins = n_bins
        self.quantiles = torch.tensor([-1.0, -0.696, -0.525, -0.395, -0.284,
                                        -0.185, -0.093, 0.0, 0.093, 0.185,
                                        0.284, 0.395, 0.525, 0.696, 1.0])

    def quantize(self, tensor):
        scale = tensor.abs().max() / 1.0
        scale = torch.clamp(scale, min=1e-8)
        normalized = tensor / scale
        indices = torch.zeros_like(normalized, dtype=torch.long)
        for i in range(len(self.quantiles) - 1):
            mask = (normalized >= self.quantiles[i]) & (normalized < self.quantiles[i + 1])
            indices[mask] = i
        indices[normalized >= self.quantiles[-1]] = len(self.quantiles) - 1
        return indices.to(torch.uint8), scale

    def dequantize(self, indices, scale):
        return self.quantiles[indices.long()] * scale

class QLoRALinear(nn.Module):
    """QLoRA线性层：NF4量化权重 + LoRA适配器"""
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.scaling = alpha / rank

        self.nf4 = NF4Quantizer()
        weight = torch.randn(out_features, in_features) * 0.02
        self.register_buffer('weight_indices', torch.zeros(out_features, in_features, dtype=torch.uint8))
        self.register_buffer('weight_scale', torch.tensor(1.0))
        self.weight_indices, self.weight_scale = self.nf4.quantize(weight)

        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        self.merged = False

    def forward(self, x):
        weight_deq = self.nf4.dequantize(self.weight_indices, self.weight_scale)
        result = F.linear(x, weight_deq)
        if not self.merged:
            result += (x @ self.lora_A @ self.lora_B) * self.scaling
        return result

    def merge(self):
        weight_deq = self.nf4.dequantize(self.weight_indices, self.weight_scale)
        merged_weight = weight_deq + (self.lora_A @ self.lora_B).T * self.scaling
        self.weight_indices, self.weight_scale = self.nf4.quantize(merged_weight)
        self.merged = True

qlora_layer = QLoRALinear(128, 64, rank=4)
x = torch.randn(2, 128)

with torch.no_grad():
    out_before = qlora_layer(x)
    qlora_layer.merge()
    out_after = qlora_layer(x)
    diff = (out_before - out_after).abs().max()

base_weight_mem = 128 * 64 * 4
nf4_mem = 128 * 64 * 0.5
lora_mem = (128 * 4 + 4 * 64) * 4

print(f"=== QLoRA: 量化基座 + LoRA微调 ===")
print(f"基座权重: {128}×{64} = {128*64:,} 参数")
print(f"LoRA参数: rank={4}, A={128}×{4}, B={4}×{64} = {128*4+4*64:,} 参数")
print(f"LoRA占比: {(128*4+4*64)/(128*64)*100:.2f}%")
print(f"\n内存对比:")
print(f"  FP32基座: {base_weight_mem/1024:.1f} KB")
print(f"  NF4基座: {nf4_mem/1024:.1f} KB (节省{(1-nf4_mem/base_weight_mem)*100:.0f}%)")
print(f"  LoRA适配器(FP32): {lora_mem/1024:.1f} KB")
print(f"  QLoRA总计: {(nf4_mem+lora_mem)/1024:.1f} KB")
print(f"  vs FP32全量微调: {(nf4_mem+lora_mem)/base_weight_mem*100:.1f}%")
print(f"\n合并前后输出差异: {diff:.6f} (量化误差)")
print(f"\nQLoRA核心优势:")
print(f"  1. NF4量化: 正态分布最优4bit格式，比均匀INT4精度更高")
print(f"  2. 双重量化: 量化scale本身也量化，进一步节省内存")
print(f"  3. 分页优化器: 优化器状态在CPU/GPU间分页，降低峰值显存")

### 产业级实战：使用 HuggingFace PEFT 库

生产环境中，LoRA/QLoRA 微调使用 `peft` 库，它是 HuggingFace 生态的标准 PEFT 实现，支持 LoRA、AdaLoRA、IA³、Prefix Tuning 等多种方法，与 `transformers` 和 `bitsandbytes` 无缝集成。

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import gc

MODEL_NAME = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32,
)
base_model.eval()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['c_attn'],
    bias='none',
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

total_params = sum(p.numel() for p in peft_model.parameters())
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"\n总参数: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")
print(f"可训练比例: {trainable_params/total_params*100:.2f}%")

print(f"\nLoRA配置:")
print(f"  rank (r): {lora_config.r}")
print(f"  alpha: {lora_config.lora_alpha}")
print(f"  scaling: {lora_config.lora_alpha / lora_config.r}")
print(f"  target_modules: {lora_config.target_modules}")
print(f"  dropout: {lora_config.lora_dropout}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

peft_model.train()
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=1e-4)

train_texts = [
    'The future of AI is bright and promising.',
    'Machine learning models are getting smaller and faster.',
    'On-device AI enables privacy-preserving inference.',
    'Quantization reduces model size with minimal accuracy loss.',
    'Edge computing brings AI closer to the user.',
]

encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=32, return_tensors='pt')
dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
loader = DataLoader(dataset, batch_size=2, shuffle=True)

peft_model.print_trainable_parameters()
print(f"\n开始LoRA微调...")
for epoch in range(3):
    total_loss = 0
    for batch_ids, batch_mask in loader:
        outputs = peft_model(input_ids=batch_ids, attention_mask=batch_mask, labels=batch_ids)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}/3, Loss: {total_loss/len(loader):.4f}")

peft_model.eval()
test_input = tokenizer('The future of AI is', return_tensors='pt')
with torch.no_grad():
    output = peft_model.generate(**test_input, max_new_tokens=15, do_sample=False)
print(f"\n微调后输出: {tokenizer.decode(output[0], skip_special_tokens=True)}")

In [ ]:
print(f"=== LoRA权重合并 (Merge) ===")
merged_model = peft_model.merge_and_unload()

test_input = tokenizer('The future of AI is', return_tensors='pt')
with torch.no_grad():
    output_merged = merged_model.generate(**test_input, max_new_tokens=15, do_sample=False)
    output_peft = peft_model.generate(**test_input, max_new_tokens=15, do_sample=False)

text_merged = tokenizer.decode(output_merged[0], skip_special_tokens=True)
text_peft = tokenizer.decode(output_peft[0], skip_special_tokens=True)
print(f"合并后输出: {text_merged}")
print(f"合并前输出: {text_peft}")
print(f"输出一致: {text_merged == text_peft}")

merged_params = sum(p.numel() for p in merged_model.parameters())
merged_mem = sum(p.numel() * p.element_size() for p in merged_model.parameters()) / 1024 / 1024
print(f"\n合并后参数量: {merged_params:,} (与原始模型相同)")
print(f"合并后内存: {merged_mem:.1f} MB")
print(f"\n产业界LoRA工作流:")
print(f"1. 训练: base_model + LoRA adapters (仅训练adapter参数)")
print(f"2. 保存: adapter_model.bin (仅几MB) + 可选保存合并后完整模型")
print(f"3. 部署: merge_and_unload() → 无额外推理开销")
print(f"4. 多任务: 同一base_model + 不同adapter → 按需切换")

del merged_model, peft_model, base_model
gc.collect()

In [ ]:
print(f"=== 产业级QLoRA: bitsandbytes NF4 + PEFT LoRA ===")

nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=nf4_config,
    device_map='auto' if torch.cuda.is_available() else None,
)

qlora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['c_attn'],
    bias='none',
)

qlora_model = get_peft_model(base_model_4bit, qlora_config)
qlora_model.print_trainable_parameters()

base_mem = sum(p.numel() * p.element_size() for p in base_model_4bit.parameters()) / 1024 / 1024
trainable = sum(p.numel() for p in qlora_model.parameters() if p.requires_grad)
print(f"\n基座模型(NF4)内存: {base_mem:.1f} MB")
print(f"可训练参数: {trainable:,}")
print(f"\nQLoRA产业级工作流:")
print(f"1. 加载: AutoModel.from_pretrained + BitsAndBytesConfig(load_in_4bit=True)")
print(f"2. 配置: LoraConfig + get_peft_model → 仅LoRA参数可训练")
print(f"3. 训练: SFTTrainer / Trainer → 标准训练循环")
print(f"4. 保存: model.save_pretrained() → 仅保存adapter权重")
print(f"5. 部署: merge_and_unload() → 重新量化或直接部署")

del qlora_model, base_model_4bit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()